In [1]:
import anndata as ad
import numpy as np
import json
import scipy.sparse as sp
from pathlib import Path

np.random.seed(0)

gene_path = Path("/work3/s252608/DL_project/data/raw/bulk_processed_genes.h5ad")
tx_path = Path("/work3/s252608/DL_project/data/raw/bulk_processed_transcripts.h5ad")
mapping_path = Path("/work3/s252608/DL_project/data/gene_to_transcripts.json")
out_dir = Path("/work3/s252608/DL_project/data/mock")
out_dir.mkdir(parents=True, exist_ok=True)

adata_gene = ad.read_h5ad(gene_path, backed="r")
adata_tx = ad.read_h5ad(tx_path, backed="r")

with open(mapping_path, "r") as f:
    mapping = json.load(f)

candidate_genes = [
    g for g, txs in mapping.items()
    if (g in adata_gene.var_names) and (2 <= len(txs) <= 10)
]

n_genes = min(100, len(candidate_genes))
genes_selected = np.random.choice(candidate_genes, size=n_genes, replace=False).tolist()

n_samples = min(200, adata_gene.n_obs)
sample_idx = np.sort(np.random.choice(adata_gene.n_obs, size=n_samples, replace=False))

tx_needed = []
for g in genes_selected:
    tx_needed.extend(mapping[g])

tx_needed = list(dict.fromkeys(tx_needed))
tx_final = [t for t in tx_needed if t in adata_tx.var_names]

gene_name_to_idx = {name: i for i, name in enumerate(adata_gene.var_names)}
tx_name_to_idx = {name: i for i, name in enumerate(adata_tx.var_names)}

gene_idx = np.sort(np.array([gene_name_to_idx[g] for g in genes_selected], dtype=int))
tx_idx = np.sort(np.array([tx_name_to_idx[t] for t in tx_final], dtype=int))

print("samples:", len(sample_idx))
print("genes:", len(gene_idx))
print("tx:", len(tx_idx))

# Gene subset
adata_gene_sub = adata_gene[sample_idx, gene_idx].to_memory()

# Transcript subset: row-by-row to avoid backed fancy indexing blowup
tx_rows = []
for i, r in enumerate(sample_idx):
    row = adata_tx[r:r+1, :].X[:, tx_idx]   # first slice row, then cols
    if not sp.issparse(row):
        row = sp.csr_matrix(row)
    else:
        row = row.tocsr()
    tx_rows.append(row)
    
    if (i + 1) % 50 == 0 or (i + 1) == len(sample_idx):
        print(f"Processed {i+1}/{len(sample_idx)} transcript rows")

X_tx_sub = sp.vstack(tx_rows, format="csr")

adata_tx_sub = ad.AnnData(
    X=X_tx_sub,
    obs=adata_tx.obs.iloc[sample_idx].copy(),
    var=adata_tx.var.iloc[tx_idx].copy(),
)

mapping_sub = {
    g: [t for t in mapping[g] if t in adata_tx_sub.var_names]
    for g in adata_gene_sub.var_names
}

adata_gene_sub.write_h5ad(out_dir / "bulk_mock_genes.h5ad")
adata_tx_sub.write_h5ad(out_dir / "bulk_mock_transcripts.h5ad")

with open(out_dir / "bulk_mock_gene_to_transcripts.json", "w") as f:
    json.dump(mapping_sub, f, indent=2)

print("Done.")
print("gene subset:", adata_gene_sub.shape)
print("tx subset:", adata_tx_sub.shape)

samples: 200
genes: 100
tx: 471
Processed 50/200 transcript rows
Processed 100/200 transcript rows
Processed 150/200 transcript rows
Processed 200/200 transcript rows
Done.
gene subset: (200, 100)
tx subset: (200, 471)
